# Documentation Cache and Drift Detection

**Notebook Version**: 1.0.0  
**Compatible with DBMCP**: 0.1.0+  
**Last Updated**: 2026-01-26  
**Test Database Version**: 1.0  
**Estimated Time**: 20 minutes  
**Difficulty**: Intermediate

## Overview

This notebook demonstrates how to cache database documentation to avoid repeating expensive discovery queries. The caching system:

- Exports database metadata as markdown files
- Loads cached documentation without querying the database
- Detects schema drift between cache and current state
- Supports incremental updates when drift is detected

## Prerequisites

- Completed [01_basic_connection.ipynb](01_basic_connection.ipynb) or familiar with database connections
- Understanding of database metadata concepts

## What You'll Learn

- How to export database documentation to markdown files
- How to load and use cached documentation
- How to detect and handle schema drift
- Best practices for documentation caching

## Environment Verification

In [1]:
import sys
from pathlib import Path

print(f"Python version: {sys.version}")

# Find repository root and add to path
def find_repo_root():
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    if current.name == "notebooks":
        return current.parent.parent
    return current

repo_root = find_repo_root()
sys.path.insert(0, str(repo_root))
print(f"Repository root: {repo_root}")

from examples.shared.notebook_helpers import verify_notebook_environment

verify_notebook_environment()

Python version: 3.13.1 (main, Dec 19 2024, 14:22:59) [Clang 18.1.8 ]
Repository root: /Users/jsmith79/Documents/Projects/Ongoing/dbmcp


✓ All dependencies available


True

## Section 1: Setup and Connection

First, let's connect to our test database and set up the documentation storage.

In [2]:
import shutil
import tempfile

from sqlalchemy import create_engine

from examples.shared.notebook_helpers import print_table
from src.cache.drift import DriftDetector
from src.cache.storage import DocumentationStorage
from src.db.metadata import MetadataService

# Connect to test database
test_db_path = repo_root / "examples" / "test_database" / "example.db"
engine = create_engine(f"sqlite:///{test_db_path}")

# Create a temporary directory for our cache (for demo purposes)
temp_cache_dir = Path(tempfile.mkdtemp(prefix="dbmcp_cache_"))
storage = DocumentationStorage(base_dir=temp_cache_dir)

# Create metadata service
metadata_svc = MetadataService(engine)

print("Documentation Storage ready")
print(f"Cache directory: {temp_cache_dir}")
print(f"Connected to: {test_db_path.name}")

Documentation Storage ready
Cache directory: /var/folders/34/x2zlvqvj5f17w9h56rblg3m40000gp/T/dbmcp_cache_jmc2klo3
Connected to: example.db


## Section 2: Gathering Metadata for Export

Before exporting documentation, we need to gather all the metadata from the database.
This is a one-time cost that pays off in reduced queries during future sessions.

In [3]:
# Create a test connection_id (normally this comes from connect_database)
connection_id = "test_connection_123"

# List schemas
schemas = metadata_svc.list_schemas(connection_id=connection_id)
print(f"Found {len(schemas)} schemas")

# Get tables for each schema
tables_dict = {}
columns_dict = {}
indexes_dict = {}

for schema in schemas:
    tables, _ = metadata_svc.list_tables(
        schema_name=schema.schema_name,
        connection_id=connection_id
    )
    tables_dict[schema.schema_name] = tables

    for table in tables:
        columns_dict[table.table_id] = metadata_svc.get_columns(
            table.table_name, schema.schema_name
        )
        indexes_dict[table.table_id] = metadata_svc.get_indexes(
            table.table_name, schema.schema_name
        )

total_tables = sum(len(t) for t in tables_dict.values())
total_columns = sum(len(c) for c in columns_dict.values())

print("\n=== Metadata Summary ===")
print(f"Schemas: {len(schemas)}")
print(f"Tables: {total_tables}")
print(f"Columns: {total_columns}")

Found 1 schemas

=== Metadata Summary ===
Schemas: 1
Tables: 6
Columns: 32


## Section 3: Exporting Documentation

Now let's export the documentation to markdown files. This creates a structured
cache that can be loaded without querying the database.

In [4]:
# Export documentation
cache = storage.export_documentation(
    connection_id=connection_id,
    database_name="example.db",
    schemas=schemas,
    tables=tables_dict,
    columns=columns_dict,
    indexes=indexes_dict,
    declared_fks=[],  # No declared FKs in our test DB
    inferred_fks=[],  # We'll skip inference for this demo
    include_sample_data=False,
    include_inferred_relationships=False,
)

print("Documentation exported!")
print(f"  Cache directory: {cache.cache_dir}")
print(f"  Schema hash: {cache.schema_hash[:16]}...")

# List the created files
cache_path = Path(cache.cache_dir)
print("\n=== Files Created ===")
for file in sorted(cache_path.rglob("*.md")):
    rel_path = file.relative_to(cache_path)
    size = file.stat().st_size
    print(f"  {rel_path}: {size:,} bytes")

Documentation exported!
  Cache directory: /var/folders/34/x2zlvqvj5f17w9h56rblg3m40000gp/T/dbmcp_cache_jmc2klo3/test_connection_123
  Schema hash: 186635c41bb32710...

=== Files Created ===
  overview.md: 369 bytes
  relationships.md: 144 bytes
  schemas/main.md: 548 bytes
  tables/main.customers.md: 456 bytes
  tables/main.order_items.md: 441 bytes
  tables/main.orders.md: 456 bytes
  tables/main.product_reviews.md: 493 bytes
  tables/main.products.md: 436 bytes
  tables/main.shipping_addresses.md: 495 bytes


## Section 4: Examining the Generated Documentation

Let's look at what the generated markdown files contain.

In [5]:
# Read the overview file
overview_path = cache_path / "overview.md"
overview_content = overview_path.read_text()

print("=== overview.md ===")
print(overview_content[:1000])
if len(overview_content) > 1000:
    print(f"\n... ({len(overview_content) - 1000} more characters)")

=== overview.md ===
# Database: example.db

**Generated**: 2026-02-02T11:47:53.727737

## Summary

- **Schemas**: 1
- **Tables**: 6
- **Declared Foreign Keys**: 0
- **Inferred Relationships**: 0

## Schemas

| Schema | Tables | Views |
|--------|--------|-------|
| [main](schemas/main.md) | 6 | 0 |

## Quick Navigation

- [Relationships](relationships.md) - All foreign key relationships


In [6]:
# Read a table file to see detailed documentation
tables_dir = cache_path / "tables"
table_files = list(tables_dir.glob("*.md"))

if table_files:
    sample_table = table_files[0]
    table_content = sample_table.read_text()

    print(f"=== {sample_table.name} ===")
    print(table_content[:1500])
    if len(table_content) > 1500:
        print(f"\n... ({len(table_content) - 1500} more characters)")

=== main.shipping_addresses.md ===
# Table: main.shipping_addresses

**Type**: Table
**Row Count**: 12
**Primary Key**: Yes

## Columns

| # | Column | Type | Nullable | PK | FK | Default |
|---|--------|------|----------|----|----|---------|
| 1 | `address_id` | INTEGER | Yes | Yes |  |  |
| 2 | `customer_id` | INTEGER | No |  |  |  |
| 3 | `street` | VARCHAR(255) | Yes |  |  |  |
| 4 | `city` | VARCHAR(100) | Yes |  |  |  |
| 5 | `postal_code` | VARCHAR(20) | Yes |  |  |  |
| 6 | `country` | VARCHAR(2) | Yes |  |  | 'US' |


## Section 5: Loading Cached Documentation

In a future session, you can load the cached documentation without querying the database.
This significantly reduces token usage and query time.

In [7]:
# Simulate a new session - create a fresh storage instance
storage_new = DocumentationStorage(base_dir=temp_cache_dir)

# Check if cache exists
print(f"Cache exists: {storage_new.cache_exists(connection_id)}")

# Load cached documentation
cached = storage_new.load_cached_docs(connection_id)

print("\n=== Loaded Cache Summary ===")
print(f"Database: {cached['database_name']}")
print(f"Cache age: {cached['cache_age_days']} days")
print("\nEntity counts:")
for entity, count in cached['entity_counts'].items():
    print(f"  {entity}: {count}")

Cache exists: True

=== Loaded Cache Summary ===
Database: example.db
Cache age: 0 days

Entity counts:
  schemas: 1
  tables: 6
  declared_fks: 0
  inferred_fks: 0


In [8]:
# Access the cached schema information
print("=== Cached Schemas ===")
results = []
for schema in cached['schemas']:
    results.append([
        schema['schema_name'],
        str(schema['table_count']),
        str(schema['view_count'])
    ])

print_table(results, ["Schema", "Tables", "Views"])

=== Cached Schemas ===
Schema | Tables | Views
-----------------------
main   | 6      | 0    

Total: 1 rows


## Section 6: Schema Drift Detection

The drift detector compares cached documentation with the current database state.
This helps identify when the cache needs to be refreshed.

In [9]:
# Create drift detector
detector = DriftDetector(storage_new)

# Check for drift (should be no drift since we just created the cache)
drift_result = detector.check_drift(
    connection_id=connection_id,
    current_schemas=schemas,
    current_tables=tables_dict,
)

print("=== Drift Check Results ===")
print(f"Drift detected: {drift_result.drift_detected}")
print(f"Summary: {drift_result.summary}")
print("\nHash comparison:")
print(f"  Cached:  {drift_result.cached_hash[:32]}...")
print(f"  Current: {drift_result.current_hash[:32]}...")

=== Drift Check Results ===
Drift detected: False
Summary: No changes detected

Hash comparison:
  Cached:  186635c41bb32710c08a16d9e6f49b9f...
  Current: 186635c41bb32710c08a16d9e6f49b9f...


## Section 7: Simulating Schema Changes

Let's simulate what happens when the database schema changes.

In [10]:
from src.models.schema import Table

# Simulate adding a new table
modified_tables = dict(tables_dict)  # Copy the dict
if 'main' in modified_tables:
    # Add a fake new table
    new_table = Table(
        table_id="main.new_audit_log",
        schema_id="main",
        table_name="new_audit_log",
        row_count=0,
    )
    modified_tables['main'] = list(modified_tables['main']) + [new_table]

# Check for drift with the modified schema
drift_result = detector.check_drift(
    connection_id=connection_id,
    current_schemas=schemas,
    current_tables=modified_tables,
)

print("=== Drift Detection After Schema Change ===")
print(f"Drift detected: {drift_result.drift_detected}")
print(f"Summary: {drift_result.summary}")

if drift_result.added_tables:
    print("\nAdded tables:")
    for table in drift_result.added_tables:
        print(f"  + {table}")

if drift_result.removed_tables:
    print("\nRemoved tables:")
    for table in drift_result.removed_tables:
        print(f"  - {table}")

=== Drift Detection After Schema Change ===
Drift detected: True
Summary: Added tables: main.new_audit_log

Added tables:
  + main.new_audit_log


In [11]:
# Generate a human-readable drift report
report = detector.generate_drift_report(drift_result)
print(report)

# Schema Drift Report

**Checked**: 2026-02-02T11:47:53.748853
**Drift Detected**: Yes

## Summary

Added tables: main.new_audit_log

## Added Tables

- `main.new_audit_log`

## Recommendation

Run `export_documentation` to update the cached documentation.


## Section 8: Cache Workflow Best Practices

Here's a recommended workflow for using the documentation cache effectively.

In [12]:
def cache_workflow_example(connection_id: str, metadata_svc, storage):
    """
    Example workflow for using documentation cache.
    
    Returns cached data or fresh data if cache is stale.
    """
    # Step 1: Check if cache exists
    if not storage.cache_exists(connection_id):
        print("No cache found - full discovery required")
        return None, "no_cache"

    # Step 2: Load cached documentation
    cached = storage.load_cached_docs(connection_id)
    cache_age = cached.get('cache_age_days', 0)

    # Step 3: Check cache freshness
    MAX_CACHE_AGE_DAYS = 7  # Refresh weekly

    if cache_age and cache_age > MAX_CACHE_AGE_DAYS:
        print(f"Cache is {cache_age} days old - refresh recommended")
        return cached, "stale"

    # Step 4: (Optional) Check for drift
    # This requires querying the database, so only do if needed

    print(f"Using cached documentation ({cache_age} days old)")
    return cached, "fresh"


# Demonstrate the workflow
result, status = cache_workflow_example(connection_id, metadata_svc, storage_new)
print(f"\nStatus: {status}")
if result:
    print(f"Loaded {result['entity_counts']['tables']} tables from cache")

Using cached documentation (0 days old)

Status: fresh
Loaded 6 tables from cache


## Cleanup

In [13]:
# Clean up the temporary cache directory
shutil.rmtree(temp_cache_dir)
print(f"Cleaned up temporary cache: {temp_cache_dir}")

Cleaned up temporary cache: /var/folders/34/x2zlvqvj5f17w9h56rblg3m40000gp/T/dbmcp_cache_jmc2klo3


## Summary

**What we covered**:
- Exported database documentation to markdown files
- Examined the structure of generated documentation
- Loaded cached documentation without database queries
- Detected schema drift between cache and current state
- Generated drift reports for change analysis

**Key concepts**:
- **Documentation Cache**: Markdown files storing database metadata
- **Schema Hash**: SHA-256 hash of schema structure for drift detection
- **Drift Detection**: Comparison of cached vs current schema state
- **Cache Workflow**: Check existence, Load, Validate freshness, Check drift

**Benefits of caching**:
- Reduced database queries in subsequent sessions
- Lower token usage for AI agents
- Human-readable documentation as a side effect
- Drift detection alerts for schema changes

**Best practices**:
- Refresh cache periodically (e.g., weekly)
- Check for drift when schema changes are expected
- Include sample data for columns that need interpretation
- Don't include sensitive data in cached documentation

## Next Steps

**Continue learning**:
- [06_query_execution.ipynb](06_query_execution.ipynb) - Execute SQL queries safely
- [03_advanced_patterns.ipynb](03_advanced_patterns.ipynb) - Relationship inference and error handling

**Explore further**:
- Try exporting documentation with sample data included
- Set up automated drift detection in your workflow
- Use cached documentation to reduce AI token usage

**Need help?**
- [Report issues](https://github.com/anthropics/dbmcp/issues)
- [Join discussions](https://github.com/anthropics/dbmcp/discussions)